## Configurazione

In [ ]:
import os
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import itertools
import math
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random
import glob


#PATH per i relativi dataset
MVTEC_DIR = '/content/drive/MyDrive/Project Work CV/MVTec1'


#Categorie da codificare
LABELS = ['leather', 'pill', 'tile', 'transistor', 'wood']


# Dataset

## Caricamento dataset MVTec da google drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
class RGB_to_BGR(object):
    def __call__(self, tensor):
        return tensor[[2,1,0], ...]

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    RGB_to_BGR()
])

## Dataset di test

In [ ]:
#Uso una classe perché non posso usare ImageFolder
#ImgaeFolder non è in grado di navigare nelle sottocartelle di test
class MVTecTestDataset(Dataset):
    def __init__(self, root_dir, category, transform):
        self.transform = transform
        #tutti i path delle immagini in test
        self.image_paths = []

        #percorso della cartella di test per la specifica categoria
        test_dir = os.path.join(root_dir, category, 'test')

        self.image_paths = glob.glob(os.path.join(test_dir, "*", "*.png"))

    #per il dataloader
    def __len__(self):
        return len(self.image_paths)

    #funzione per ottenere l'immagine
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]

        #prendo tutte le sotto cartelle di test (tipi di anomalia)
        subfolders = img_path.split(os.sep)
        img_type = subfolders[-2]
        #nome immagine
        img_name = subfolders[-1]
        #nome categoria
        category = subfolders[-4]

        #applicazione transform all'immagine per resize e BGR
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        #se l'immagine non ha anomalia --> label = 0
        #se l'immagine ha un qualsiasi tipo di anomalia --> label 1
        if img_type == 'good':
            label = 0
            mask = torch.zeros((1, image.shape[1], image.shape[2]))
        else:
            label = 1 
            #prendo la maschera ground_truth dell'immagine corrispondente
            root_path = os.sep.join(subfolders[:-4])
            mask_filename = img_name.replace(".png", "_mask.png")
            mask_path = os.path.join(root_path, category, 'ground_truth', img_type, mask_filename)

            #converto la maschera in scala di grigi
            mask = Image.open(mask_path).convert('L')

            
            mask_transform = transforms.Compose([
                transforms.Resize((256, 256)),
                transforms.ToTensor()
            ])
            mask = mask_transform(mask)

        return image, mask, label

# Modello DRAEM - FPN

In [ ]:
import torch
import torch.nn as nn


class ReconstructiveSubNetwork(nn.Module):
    def __init__(self,in_channels=3, out_channels=3, base_width=128):
        super(ReconstructiveSubNetwork, self).__init__()
        self.encoder = EncoderReconstructive(in_channels, base_width)
        self.decoder = DecoderReconstructive(base_width, out_channels=out_channels)

    def forward(self, x):
        b5 = self.encoder(x)
        output = self.decoder(b5)
        return output

    def getName(self):
        return self.__class__.__name__


class EncoderReconstructive(nn.Module):
    def __init__(self, in_channels, base_width):
        super(EncoderReconstructive, self).__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels,base_width, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width, base_width, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width),
            nn.ReLU(inplace=True))
        self.mp1 = nn.Sequential(nn.MaxPool2d(2))
        self.block2 = nn.Sequential(
            nn.Conv2d(base_width,base_width*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*2),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width*2, base_width*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*2),
            nn.ReLU(inplace=True))
        self.mp2 = nn.Sequential(nn.MaxPool2d(2))
        self.block3 = nn.Sequential(
            nn.Conv2d(base_width*2,base_width*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*4),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width*4, base_width*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*4),
            nn.ReLU(inplace=True))
        self.mp3 = nn.Sequential(nn.MaxPool2d(2))
        self.block4 = nn.Sequential(
            nn.Conv2d(base_width*4,base_width*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*8),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width*8, base_width*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*8),
            nn.ReLU(inplace=True))
        self.mp4 = nn.Sequential(nn.MaxPool2d(2))
        self.block5 = nn.Sequential(
            nn.Conv2d(base_width*8,base_width*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*8),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width*8, base_width*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*8),
            nn.ReLU(inplace=True))


    def forward(self, x):
        b1 = self.block1(x)
        mp1 = self.mp1(b1)
        b2 = self.block2(mp1)
        mp2 = self.mp3(b2)
        b3 = self.block3(mp2)
        mp3 = self.mp3(b3)
        b4 = self.block4(mp3)
        mp4 = self.mp4(b4)
        b5 = self.block5(mp4)
        return b5


class DecoderReconstructive(nn.Module):
    def __init__(self, base_width, out_channels=1):
        super(DecoderReconstructive, self).__init__()

        self.up1 = nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
                                 nn.Conv2d(base_width * 8, base_width * 8, kernel_size=3, padding=1),
                                 nn.BatchNorm2d(base_width * 8),
                                 nn.ReLU(inplace=True))
        self.db1 = nn.Sequential(
            nn.Conv2d(base_width*8, base_width*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*8),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width * 8, base_width * 4, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width * 4),
            nn.ReLU(inplace=True)
        )

        self.up2 = nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
                                 nn.Conv2d(base_width * 4, base_width * 4, kernel_size=3, padding=1),
                                 nn.BatchNorm2d(base_width * 4),
                                 nn.ReLU(inplace=True))
        self.db2 = nn.Sequential(
            nn.Conv2d(base_width*4, base_width*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*4),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width * 4, base_width * 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width * 2),
            nn.ReLU(inplace=True)
        )

        self.up3 = nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
                                 nn.Conv2d(base_width * 2, base_width*2, kernel_size=3, padding=1),
                                 nn.BatchNorm2d(base_width*2),
                                 nn.ReLU(inplace=True))
        # cat with base*1
        self.db3 = nn.Sequential(
            nn.Conv2d(base_width*2, base_width*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*2),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width*2, base_width*1, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*1),
            nn.ReLU(inplace=True)
        )

        self.up4 = nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
                                 nn.Conv2d(base_width, base_width, kernel_size=3, padding=1),
                                 nn.BatchNorm2d(base_width),
                                 nn.ReLU(inplace=True))
        self.db4 = nn.Sequential(
            nn.Conv2d(base_width*1, base_width, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width, base_width, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width),
            nn.ReLU(inplace=True)
        )

        self.fin_out = nn.Sequential(nn.Conv2d(base_width, out_channels, kernel_size=3, padding=1))
        #self.fin_out = nn.Conv2d(base_width, out_channels, kernel_size=3, padding=1)

    def forward(self, b5):
        up1 = self.up1(b5)
        db1 = self.db1(up1)

        up2 = self.up2(db1)
        db2 = self.db2(up2)

        up3 = self.up3(db2)
        db3 = self.db3(up3)

        up4 = self.up4(db3)
        db4 = self.db4(up4)

        out = self.fin_out(db4)
        return out

## FPN

In [ ]:
class EncoderDiscriminative(nn.Module):
    def __init__(self, in_channels, base_width):
        super(EncoderDiscriminative, self).__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels,base_width, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width, base_width, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width),
            nn.ReLU(inplace=True))
        self.mp1 = nn.Sequential(nn.MaxPool2d(2))
        self.block2 = nn.Sequential(
            nn.Conv2d(base_width,base_width*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*2),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width*2, base_width*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*2),
            nn.ReLU(inplace=True))
        self.mp2 = nn.Sequential(nn.MaxPool2d(2))
        self.block3 = nn.Sequential(
            nn.Conv2d(base_width*2,base_width*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*4),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width*4, base_width*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*4),
            nn.ReLU(inplace=True))
        self.mp3 = nn.Sequential(nn.MaxPool2d(2))
        self.block4 = nn.Sequential(
            nn.Conv2d(base_width*4,base_width*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*8),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width*8, base_width*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*8),
            nn.ReLU(inplace=True))
        self.mp4 = nn.Sequential(nn.MaxPool2d(2))
        self.block5 = nn.Sequential(
            nn.Conv2d(base_width*8,base_width*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*8),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width*8, base_width*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*8),
            nn.ReLU(inplace=True))

        self.mp5 = nn.Sequential(nn.MaxPool2d(2))
        self.block6 = nn.Sequential(
            nn.Conv2d(base_width*8,base_width*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*8),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width*8, base_width*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*8),
            nn.ReLU(inplace=True))


    def forward(self, x):
        b1 = self.block1(x)
        mp1 = self.mp1(b1)
        b2 = self.block2(mp1)
        mp2 = self.mp2(b2)
        b3 = self.block3(mp2)
        mp3 = self.mp3(b3)
        b4 = self.block4(mp3)
        mp4 = self.mp4(b4)
        b5 = self.block5(mp4)
        mp5 = self.mp5(b5)
        b6 = self.block6(mp5)
        return b1,b2,b3,b4,b5,b6


#6 tensori in uscita come i blocchi dell'encoder
#non devo generare le mappe di feature a diversa scala perchè uso già quelle dell'encoder
#quindi parto dalle convoluzione laterali per ridimensionare in profondità
class DecoderFPN(nn.Module):
    def __init__(self, base_width=64, fpn_channels=256, out_channels=2):
        super(DecoderFPN, self).__init__()
        #conv 1x1
        self.depth_conv6 = nn.Conv2d(base_width * 8, fpn_channels, kernel_size=1)
        self.depth_conv5 = nn.Conv2d(base_width * 8, fpn_channels, kernel_size=1)
        self.depth_conv4 = nn.Conv2d(base_width * 8, fpn_channels, kernel_size=1)
        self.depth_conv3 = nn.Conv2d(base_width * 4, fpn_channels, kernel_size=1)
        self.depth_conv2 = nn.Conv2d(base_width * 2, fpn_channels, kernel_size=1)
        self.depth_conv1 = nn.Conv2d(base_width * 1, fpn_channels, kernel_size=1)

        #convluzioni 3x3 di smoothing per eliminare errori di upsampling
        self.smooth6 = nn.Conv2d(fpn_channels, fpn_channels, kernel_size=3, padding=1)
        self.smooth5 = nn.Conv2d(fpn_channels, fpn_channels, kernel_size=3, padding=1)
        self.smooth4 = nn.Conv2d(fpn_channels, fpn_channels, kernel_size=3, padding=1)
        self.smooth3 = nn.Conv2d(fpn_channels, fpn_channels, kernel_size=3, padding=1)
        self.smooth2 = nn.Conv2d(fpn_channels, fpn_channels, kernel_size=3, padding=1)
        self.smooth1 = nn.Conv2d(fpn_channels, fpn_channels, kernel_size=3, padding=1)

        #Output
        #fpn_channels * 6 perchè riceve sei mappe di feature
        self.fin_conv = nn.Sequential(
            nn.Conv2d(fpn_channels * 6, fpn_channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(fpn_channels, out_channels, kernel_size=3, padding=1)
        )

    def forward(self, b1, b2, b3, b4, b5, b6):

        #Somma mappe di feature + smoothin 3x3
        p6 = self.depth_conv6(b6)

        p5 = self.depth_conv5(b5) + F.interpolate(p6, scale_factor=2, mode='bilinear', align_corners=False)
        p5 = self.smooth5(p5)

        p4 = self.depth_conv4(b4) + F.interpolate(p5, scale_factor=2, mode='bilinear', align_corners=False)
        p4 = self.smooth4(p4)

        p3 = self.depth_conv3(b3) + F.interpolate(p4, scale_factor=2, mode='bilinear', align_corners=False)
        p3 = self.smooth3(p3)

        p2 = self.depth_conv2(b2) + F.interpolate(p3, scale_factor=2, mode='bilinear', align_corners=False)
        p2 = self.smooth2(p2)

        p1 = self.depth_conv1(b1) + F.interpolate(p2, scale_factor=2, mode='bilinear', align_corners=False)
        p1 = self.smooth1(p1)

        #Upsample a 256x256
        original_size = (256,256)

        p6_up = F.interpolate(p6, size=original_size, mode='bilinear', align_corners=False)
        p5_up = F.interpolate(p5, size=original_size, mode='bilinear', align_corners=False)
        p4_up = F.interpolate(p4, size=original_size, mode='bilinear', align_corners=False)
        p3_up = F.interpolate(p3, size=original_size, mode='bilinear', align_corners=False)
        p2_up = F.interpolate(p2, size=original_size, mode='bilinear', align_corners=False)
        p1_up = F.interpolate(p1, size=original_size, mode='bilinear', align_corners=False)

        #Concatenazione mappe
        out_concat = torch.cat((p1_up,p2_up,p3_up,p4_up,p5_up,p6_up),dim=1)

        out = self.fin_conv(out_concat)

        return out


class FPNNetwork(nn.Module):
    def __init__(self, in_channels=6, out_channels=2, base_width=64):
        super(FPNNetwork, self).__init__()
        self.encoder = EncoderDiscriminative(in_channels, base_width)
        self.decoder = DecoderFPN(base_width, fpn_channels=256, out_channels=out_channels)

    def forward(self, x):
        b1, b2, b3, b4, b5, b6 = self.encoder(x)
        fpn_output = self.decoder(b1,b2,b3,b4,b5,b6)
        return fpn_output

    def getName(self):
        return self.__class__.__name__



# Load dei pesi

In [ ]:
WEIGHT_DIR = '/content/drive/MyDrive/Project Work CV/weights'

#AUTOENCODER
AE_DIR = '/content/drive/MyDrive/Project Work CV/paper_weights'
AE_HEADER = 'DRAEM_seg_large_ae_large_0.0001_800_bs8_'
#UNET
FPN_DIR = '/content/drive/MyDrive/Project Work CV/weights/fpn'
FPN_HEADER = 'fpn_weights_'

def loadWeight(model, label_name, device):
    if(model.getName() == 'ReconstructiveSubNetwork'):
        ae_file_name = AE_HEADER + label_name + '_.pckl'
        ae_load_path = os.path.join(AE_DIR,ae_file_name)
        if os.path.exists(ae_load_path):
            model.load_state_dict(torch.load(ae_load_path, map_location=device))
            print(f"AE: Pesi per la categoria '{label_name.upper()}' caricati correttamente!\n - {ae_load_path}")
        else:
            print(f"Errore: File dei pesi non trovati.\n{ae_load_path}\n")

    if(model.getName() == 'FPNNetwork'):
        fpn_file_name = FPN_HEADER + label_name + '.pth'
        fpn_load_path = os.path.join(FPN_DIR,fpn_file_name)
        if os.path.exists(fpn_load_path):
            model.load_state_dict(torch.load(fpn_load_path, map_location=device))
            print(f"FPN: Pesi per la categoria '{label_name.upper()}' caricati correttamente!\n - {fpn_load_path}")
        else:
            print(f"FPN: Errore: File dei pesi non trovati. Verifica i percorsi:\n - {fpn_load_path}\n")

## Test

In [ ]:
if torch.cuda.is_available():
  device = torch.device('cuda')
else:
  device = torch.device('cpu')

print("using ", device)

category = 'tile'

test_dataset = MVTecTestDataset(MVTEC_DIR, category, transform)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

model = ReconstructiveSubNetwork(in_channels=3, out_channels=3, base_width= 128)
model_seg = FPNNetwork(in_channels=6, out_channels=2, base_width=64)

loadWeight(model, category, device)
loadWeight(model_seg, category, device)
model.to(device)
model_seg.to(device)
model.eval()
model_seg.eval()
pass

using  cpu
AE: Pesi per la categoria 'TILE' caricati correttamente!
 - /content/drive/MyDrive/Project Work CV/paper_weights/DRAEM_seg_large_ae_large_0.0001_800_bs8_tile_.pckl
FPN: Pesi per la categoria 'TILE' caricati correttamente!
 - /content/drive/MyDrive/Project Work CV/weights/fpn/fpn_weights_tile.pth


In [ ]:
total_parameters = sum(p.numel() for p in model_seg.parameters())

train_parameters = sum(p.numel() for p in model_seg.parameters() if p.requires_grad)

print(f"Total parameters: {total_parameters}")
print(f"Trainable parameters: {train_parameters}")

Total parameters: 21728002
Trainable parameters: 21728002


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
import numpy as np

#inizializzazione
anomaly_score_prediction = []
anomaly_score_gt = []

dim_img = 256
total_pixel_scores = np.zeros((dim_img * dim_img * len(test_dataset)))
total_gt_pixel_scores = np.zeros((dim_img * dim_img * len(test_dataset)))
mask_cnt = 0

with torch.no_grad():
    for image, mask, label in test_loader:
        image = image.to(device)

        anomaly_score_gt.append(label.item())
        true_mask_cv = mask.squeeze().numpy()

        mask = mask.to(device)

        rec_img = model(image)

        concat_img = torch.cat((rec_img, image), dim=1)

        out_mask = model_seg(concat_img)
        #probabilità maschera
        out_mask_sm = torch.softmax(out_mask, dim=1)

        out_mask_cv = out_mask_sm[0, 1, :, :].cpu().numpy()

        out_mask_averaged = torch.nn.functional.avg_pool2d(out_mask_sm[:, 1:, :, :], 21, stride=1, padding=21 // 2).cpu().numpy()
        image_score = np.max(out_mask_averaged)
        anomaly_score_prediction.append(image_score)

        flat_true_mask = true_mask_cv.flatten()
        flat_out_mask = out_mask_cv.flatten()

        total_pixel_scores[mask_cnt * dim_img * dim_img:(mask_cnt + 1) * dim_img * dim_img] = flat_out_mask
        total_gt_pixel_scores[mask_cnt * dim_img * dim_img:(mask_cnt + 1) * dim_img * dim_img] = flat_true_mask
        mask_cnt += 1


anomaly_score_prediction = np.array(anomaly_score_prediction)
anomaly_score_gt = np.array(anomaly_score_gt)


auroc_img = roc_auc_score(anomaly_score_gt, anomaly_score_prediction)


total_gt_pixel_scores = total_gt_pixel_scores.astype(np.uint8)
total_gt_pixel_scores = total_gt_pixel_scores[:dim_img * dim_img * mask_cnt]
total_pixel_scores = total_pixel_scores[:dim_img * dim_img * mask_cnt]

auroc_pixel = roc_auc_score(total_gt_pixel_scores, total_pixel_scores)
ap_pixel = average_precision_score(total_gt_pixel_scores, total_pixel_scores)

print(f"Risultati per la categoria: {category.upper()}\n")
print("Metriche: \tAUROC IMG \tAUROC PIXEL \tAVPIXEL")
print(f"Valori:  \t{auroc_img:.4f}, \t{auroc_pixel:.4f}, \t{ap_pixel:.4f}")
print(f"Percentuali:  \t{auroc_img*100}, \t{auroc_pixel*100}, \t{ap_pixel*100}")

precisions, recalls, thresholds = precision_recall_curve(total_gt_pixel_scores, total_pixel_scores)


f1_scores = (2 * precisions * recalls) / (precisions + recalls + 1e-8)


optimal_idx = np.argmax(f1_scores)

optimal_f1 = f1_scores[optimal_idx]
optimal_precision = precisions[optimal_idx]
optimal_recall = recalls[optimal_idx]
optimal_threshold = thresholds[optimal_idx] if optimal_idx < len(thresholds) else 1.0

print("Metriche: \tRECALL \tPRECISION \tF1-SCORE")
print(f"Valori: \t{optimal_recall:.4f}, \t{optimal_precision:.4f}, \t{optimal_f1:.4f}")
print(f"Percentuali: \t{optimal_recall*100}, \t{optimal_precision*100}, \t{optimal_f1*100}")



KeyboardInterrupt: 